### 연습문제
- Doc2Vec 라이브러리 이용한 감정분석
- 데이터는 rationgs_train.txt 파일을 로드
    - 특수 문자, 2칸 이상의 공백의 문자를 제거하는 정규화 함수
    - document 컬럼의 데이터에서 중복 데이터를 제거
    - 빈 텍스트, " "가 존재한다면 해당 행 데이터도 제거
    - 상위의 5000개 정도 데이터를 이용
- 토큰화 함수 komoran를 이용
    - 필요한 품사 : NNP, NNG, VV, VA, MAG, XR만을 사용
    - 불용어 단어 : 하다, 되다. 이다, 것, 수, 거
- 데이터에서 독립(document), 종속(label) 변수로 데이터를 나누고 train, test 데이터셋을 나눠준다. 비율은 8:2
- Doc2Vec 객체를 생성하여 학습
    - 매개변수
        - vector_size = 200
        - window = 5
        - min_count = 2
        - dm = 1
        - negative = 5
        - seed = 42
        - epochs = 50
    - 학습 시키는 데이터는 X_train
- X_train, X_test -> 문자열 데이터 -> infer_vector() 함수를 이용해서 임베딩
- 고전 머신러닝 분류 모델을 이용하여 임베딩된 데이터를 독립 변수로 X의 데이터들을 종속 변수로 학습하여 예측
    - 정확도를 확인
    - LogisticRegression(max_iter = 2000, radom_state = 42)
    - LinearSVC(random_state = 42)
    - 두개의 모델을 사용하여 정확도가 좋은 모델을 선택


In [112]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from konlpy.tag import Komoran
from gensim.models.doc2vec import Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import re


In [113]:
data = pd.read_csv("../data/ratings_train.txt", sep="\t")
data.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [114]:
data['document'] = data['document'].fillna('')

In [115]:
# document의 중복과 빈 텍스트, " " 행 제거
data = data.drop_duplicates(subset=['document'])
data = data[data['document'].str.strip() != '']

In [116]:
# 문자에서 불필요한 글자들을 제거 (정규화)
def nomalize(text):
    # 특수문자 제외 , 공백에 대한 처리 
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data['document'] = [nomalize(text) for text in data['document'].values]

In [117]:
data = data[:5000]

In [118]:
# 형태소 분석 Komoran을 이용하여 토큰화 
komoran = Komoran()

# 특정 품사만 사용 
allow_pos = [ 'NNP', 'NNG', 'VV', 'VA', 'MAG', 'XR' ]
# 불용어 
stop_word = ['하다', '되다', '이다', '것', '수', '거']

def tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            # 길이를 체크하기 전에 동사, 형용사 에는 '다' 붙이기 활용
            if pos in ['VV', 'VA']:
                word += '다'
            if word not in stop_word and len(word) > 1:
                # allow_pos의 포함되어있고
                # stop_word에 포함되어있지 않으며
                # 단어의 길이가 1보다 큰 문자만 활용
                tokens.append(word)
    return tokens

tokenize_docs = [ tokenize(doc) for doc in data['document'].values ]
tokenize_docs

[['더빙', '진짜', '짜증', '나다', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기', '가볍다'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없다', '평점', '조정'],
 ['익살', '연기', '돋보이다', '영화', '스파이더맨', '늙다', '보이다', '커스틴 던스트', '너무나'],
 ['걸음마', '떼다', '초등학교', '학년', '영화', '반개', '아깝다'],
 ['원작', '긴장감', '제대로', '살리다'],
 ['반개',
  '아깝다',
  '나오다',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '드라마',
  '가족',
  '없다',
  '연기',
  '못하다',
  '사람',
  '모이다'],
 ['액션', '없다', '재미', '있다', '영화'],
 ['평점', '낮다', '보다', '헐리우드', '화려', '너무', '길들이다', '있다'],
 [],
 ['눈물', '나서다', '죽다', '향수', '자극', '허진호', '감성', '절제', '멜로', '달인'],
 ['울다', '손들다', '횡단보도', '건너다', '뛰쳐나오다', '이범수', '연기', '드럽다'],
 ['담백', '깔끔', '좋다', '신문', '기사', '로만', '보다', '보다', '자꾸', '잊어버리다', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보다',
  '영화',
  '가장',
  '재다',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['사람',
  '웃기다',
  '바스코',
  '이기다',
  '락스',
  '바비',
  '이기다',
  '아이돌',
  '깔다',
  '그냥',
  '까다',
  '안달',
  '보이다'],
 ['굿바이 레닌', '표절', '이해', '갈수록', '

In [119]:
X = data['document'].values
Y = data['label'].values

In [133]:
tagged = []

for idx , toks in enumerate(tokenize_docs):
    if len(toks) > 0:
        tagged.append(
            TaggedDocument(words=toks, tags=[f"DOC_{idx}"])
        )
    else:
        tagged.append(
            TaggedDocument(words="", tags=[f"DOC_{idx}"])
        )

In [134]:
# tag을 추가한 문서를 이용해서 Doc2Vec 모델에 학습 데이터로 이용
model = Doc2Vec(
    documents= tagged, 
    vector_size= 500, 
    window= 5, 
    min_count = 2,   # 데이터의 개수가 작기 때문, 실제 3-5
    dm = 1,          # PV-DM 방식
    negative = 5,    # 잘못된 단어간의 배치를 사용하여 학습에 이용
    epochs= 50, 
    seed = 42
)

In [136]:
X = np.vstack(
    [model.dv[f"DOC_{idx}"] for idx in range(len(data['document']))]
)
X.shape

(5000, 500)

In [137]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y,
                                                    test_size=0.2, random_state=42, stratify=Y)

In [138]:
lr = LogisticRegression(max_iter=2000, random_state=42)
svc = SVC(random_state=42)

In [139]:
lr.fit(X_train, Y_train)
pred_lr = lr.predict(X_test)

In [140]:
svc.fit(X_train, Y_train)
pred_svc = svc.predict(X_test)

In [141]:
print(classification_report(Y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.73      0.75      0.74       500
           1       0.74      0.72      0.73       500

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000



In [142]:
print(classification_report(Y_test, pred_svc))

              precision    recall  f1-score   support

           0       0.71      0.81      0.76       500
           1       0.78      0.67      0.72       500

    accuracy                           0.74      1000
   macro avg       0.75      0.74      0.74      1000
weighted avg       0.75      0.74      0.74      1000

